In [13]:
import pandas as pd

In [14]:
df_price=pd.read_csv("../data/processed/combined_data.csv", parse_dates=["period_date"])

In [36]:
df_weather = pd.read_csv("data/external/ethiopia_nasa_monthly_weather_2008_2026_full.csv", parse_dates=["period_date"])

In [37]:
# 2. Force convert period_date to datetime in BOTH dataframes
df_price["period_date"] = pd.to_datetime(df_price["period_date"], errors="coerce")
df_weather["period_date"] = pd.to_datetime(df_weather["period_date"], errors="coerce")

In [38]:
# 3. Make both dates the first day of the month
df_price["period_date"] = df_price["period_date"].dt.to_period("M").dt.to_timestamp()
df_weather["period_date"] = df_weather["period_date"].dt.to_period("M").dt.to_timestamp()

In [56]:
# 3. Merge
df_merged = pd.merge(
    df_price,
    df_weather,
    on=["market", "period_date"],
    how="left"
)

print("\nMerged shape:", df_merged.shape)
print("Rows with weather:", df_merged["temp_mean"].notna().sum())
print("Rows without weather:", df_merged["temp_mean"].isna().sum())


Merged shape: (34331, 16)
Rows with weather: 33099
Rows without weather: 1232


In [57]:
# 4. Clean column names (remove _x / _y if they appear)
df_merged = df_merged.rename(columns={
    "latitude_x": "latitude",
    "longitude_x": "longitude"
})

df_merged = df_merged.drop(columns=["latitude_y", "longitude_y"], errors="ignore")

print("\nFinal columns:")
print(df_merged.columns.tolist())


Final columns:
['country', 'admin_1', 'market', 'latitude', 'longitude', 'period_date', 'currency', 'product', 'value', 'temp_mean', 'temp_max', 'temp_min', 'precipitation', 'humidity']


In [59]:
print(df_merged.isnull().sum())

country             0
admin_1             0
market              0
latitude            0
longitude           0
period_date         0
currency            0
product             0
value               0
temp_mean        1232
temp_max         1232
temp_min         1232
precipitation    1232
humidity         1232
dtype: int64


In [60]:
# 1. Check the final shape and missing values
print("Final shape:", df_merged.shape)
print("\nMissing values:")
print(df_merged.isnull().sum())

Final shape: (34331, 14)

Missing values:
country             0
admin_1             0
market              0
latitude            0
longitude           0
period_date         0
currency            0
product             0
value               0
temp_mean        1232
temp_max         1232
temp_min         1232
precipitation    1232
humidity         1232
dtype: int64


In [61]:
# 2. Decide what to do with the remaining missing weather
# Option A: Drop rows that have no weather (recommended for clean ML)
df_final = df_merged.dropna(subset=["temp_mean"]).copy()

print("Shape after dropping missing weather:", df_final.shape)

Shape after dropping missing weather: (33099, 14)


In [63]:
# 3. Final check
print("\nProduct counts:")
print(df_final["product"].value_counts())

print("\nDate range:")
print(df_final["period_date"].min(), "→", df_final["period_date"].max())

print("\nUnique markets:", df_final["market"].nunique())


Product counts:
product
maize             10524
sorghum            8380
livestock_goat     6278
wheat              4196
teff               3721
Name: count, dtype: int64

Date range:
2008-01-01 00:00:00 → 2025-12-01 00:00:00

Unique markets: 126


In [64]:
# 4. Save the clean final dataset for Machine Learning
df_final.to_csv("../data/processed/ethiopia_commodity_prices_with_weather_clean.csv", index=False)

print("\nClean dataset saved successfully!")
print("File: data/processed/ethiopia_commodity_prices_with_weather_clean.csv")


Clean dataset saved successfully!
File: data/processed/ethiopia_commodity_prices_with_weather_clean.csv
